# 导入数据


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"D:\VsCodeProjects\AB_Test_Analysis\data\ab_data.csv")
df.head()

,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


# 基本信息

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   user_id       294478 non-null  int64 
 1   timestamp     294478 non-null  object
 2   group         294478 non-null  object
 3   landing_page  294478 non-null  object
 4   converted     294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [6]:
print(df.shape)

(294478, 5)


In [8]:
print(df.isnull().sum())

user_id         0
timestamp       0
group           0
landing_page    0
converted       0
dtype: int64


In [ ]:
df["group"].value_counts()

group
treatment    147276
control      147202
Name: count, dtype: int64

 # 转化率总览

In [ ]:
df.groupby("group")["converted"].agg(["mean", "count", "sum"]).round(4)

,mean,count,sum
group,,,
control,0.1204,147202,17723
treatment,0.1189,147276,17514


# 检查重复用户

In [ ]:
dup_users = df[df.duplicated("user_id", keep=False)]
print(f"重复出现的用户数：{dup_users['user_id'].nunique()}")
print(f"重复记录总行数：{len(dup_users)}")

重复出现的用户数：3894
重复记录总行数：7788


# 检查分组与页面是否对应

In [ ]:
mismatch = df[
    ((df["group"] == "control") & (df["landing_page"] == "new_page"))
    | ((df["group"] == "treatment") & (df["landing_page"] == "old_page"))
]
print(f"分组与页面不一致的记录数：{len(mismatch)}")

分组与页面不一致的记录数：3893


# 数据清洗


In [16]:
df_clean = df[
    ~(
        ((df["group"] == "control") & (df["landing_page"] == "new_page"))
        | ((df["group"] == "treatment") & (df["landing_page"] == "old_page"))
    )
]

df_clean = df_clean.drop_duplicates(subset="user_id", keep="first")

print(f"清洗前: {len(df)} 行")
print(f"清洗后: {len(df_clean)} 行")
print(f"删除了: {len(df) - len(df_clean)} 行")

清洗前: 294478 行
清洗后: 290584 行
删除了: 3894 行


# 清洗后重新查看两组情况

In [ ]:
df_clean.groupby("group")["converted"].agg(["mean", "count", "sum"]).round(4)

,mean,count,sum
group,,,
control,0.1204,145274,17489
treatment,0.1188,145310,17264


# 保存清洗后数据

In [ ]:
df_clean.to_csv(
    r"D:\VsCodeProjects\AB_Test_Analysis\data\ab_data_clean.csv", index=False
)
print("保存成功")

保存成功
